# 04. 같은 강종의 LOT끼리 온도·출력 비교하기

[이슈 #71](https://github.com/posco-3bungi-pdm/pdm-project/issues/71) · 03번에서 LOT별로 본 뒤 읽는 비교임. 03번을 먼저 실행하지 않아도 독립 실행됨.

**같은 강종을 묶되 LOT를 평균으로 합치지 않음. 각 LOT의 선을 그대로 겹쳐 봄.**

- 가로축은 **첫 온도 관측 후 경과시간(분)**임. `60`은 첫 관측 후 1시간임. 실제 시계의 01:00을 뜻하지 않음.
- 짧은 LOT를 긴 LOT 길이에 맞춰 늘리지 않음. 기록이 끝난 뒤에는 선이 없음.
- 같은 경과시간이 같은 소재 위치나 같은 조업 단계라는 뜻은 아님.
- 같은 LOT는 모든 그림에서 같은 색임. 진한 선은 1분 평균, 옅은 색은 같은 분의 최소~최대임.
- TC 단위 ℃·OP 단위 %는 데이터사전의 가정임. CP는 단위 미확인임.

`VIEW_GRADE`만 바꿔 화면에서 볼 강종을 선택함. 7개 강종의 그림은 전부 저장함. LOT가 하나뿐인 강종은 **비교 LOT 없음**이라고 표시함.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display

# 저장소 루트나 이 노트북 폴더에서 같은 원자료를 읽기 위함
ROOT = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data" / "README.md").is_file()
    and (folder / "notebooks").is_dir()
)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from namelesspark_pipeline import load_sources, build_lot_context

sources = load_sources(ROOT)
FIGURES = ROOT / "figures" / "namelesspark" / "04_grade_comparison"

# 설치된 한글 폰트를 쓰기 위함
fonts = {font.name for font in font_manager.fontManager.ttflist}
for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font in fonts:
        plt.rcParams["font.family"] = font
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False

TARGETS = ["TC-Z3", "OP-Z3", "TC-Z4", "OP-Z4", "TC-Z5", "OP-Z5", "CP-Z4"]
raw = sources["temp"].sort_values(["LOT_NO", "MEAS_DT"])

# LOT 첫 관측과 CR1 강종변경 시각을 연결하는 공통 기준을 사용함
lots = (
    build_lot_context(raw, sources["event"])
    .set_index("LOT_NO")[["START", "END", "GRADE"]]
    .rename(columns={"START": "시작", "END": "끝", "GRADE": "강종"})
)

# LOT마다 따로 집계하므로 서로 다른 LOT 사이의 빈 날짜를 메우지 않음
# OP의 0은 실제 출력임. 결측으로 바꾸지 않음
means, lows, highs, counts = {}, {}, {}, {}
for lot, rows in raw.groupby("LOT_NO"):
    minutes = rows.set_index("MEAS_DT")[TARGETS].resample("1min")
    means[lot] = minutes.mean()
    lows[lot] = minutes.min()
    highs[lot] = minutes.max()
    counts[lot] = minutes.count()

# 아래 2개는 팀에서 확정한 해석임. 전체 LOT를 고장으로 표시하는 뜻은 아님
operation = {240058: "정기수리를 위한 정지", 240061: "정기수리 후 재가동"}


## 1. 비교할 LOT 수 확인하기

LOT가 하나뿐인 SGCC·DP590·SPCD는 강종 내에서 반복되는 현상인지 확인할 수 없음. 같은 강종이라도 운전 상태가 다를 수 있어 LOT를 따로 표시함.


In [ ]:
grade_table = lots.reset_index().groupby("강종", sort=False).agg(
    LOT수=("LOT_NO", "size"),
    LOT목록=("LOT_NO", list),
)
display(grade_table)


## 2. 같은 색의 LOT를 Z3 → Z4 → Z5 → CP 순서로 보기

예를 들어 SPHC에서 061은 재가동 LOT임. 다른 SPHC LOT와 출발 온도가 다른 이유를 먼저 구분하고 읽기. 세 구역 그림은 같은 온도축 범위를 사용하므로 구역마다 다르게 축을 늘려 보이지 않게 함.


In [ ]:
# 화면에서 볼 강종임. 다른 강종도 파일로는 모두 저장함
VIEW_GRADE = "SPHC"
COLORS = ["#2B70B2", "#D17A25", "#448A65", "#9863A5"]

for grade, group in lots.groupby("강종", sort=False):
    lot_ids = group.index.tolist()
    message = "같은 강종의 LOT 비교" if len(lot_ids) > 1 else "비교 LOT 없음"
    legend_labels = []
    for lot in lot_ids:
        label = f"{lot} ({lots.loc[lot, '시작']:%m/%d})"
        if lot in operation:
            label += " · " + operation[lot]
        legend_labels.append(label)

    # Z3·Z4·Z5 그림의 온도축 범위를 통일하기 위함
    tc_min = min(lows[lot][["TC-Z3", "TC-Z4", "TC-Z5"]].min().min() for lot in lot_ids) - 1
    tc_max = max(highs[lot][["TC-Z3", "TC-Z4", "TC-Z5"]].max().max() for lot in lot_ids) + 1

    # 위에는 온도끼리, 아래에는 출력끼리 비교함. 색은 LOT를 구분하기 위함
    for zone in [3, 4, 5]:
        fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, layout="constrained")
        for index, (lot, label) in enumerate(zip(lot_ids, legend_labels)):
            color = COLORS[index % len(COLORS)]
            mean, low, high = means[lot], lows[lot], highs[lot]
            elapsed = (mean.index - lots.loc[lot, "시작"]).total_seconds() / 60
            for ax, prefix, unit in zip(axes, ["TC", "OP"], ["온도 (℃)", "출력 (%)"]):
                sensor = f"{prefix}-Z{zone}"
                ax.plot(elapsed, mean[sensor], color=color, linewidth=1, label=label)
                ax.fill_between(elapsed, low[sensor], high[sensor], color=color, alpha=0.08)
                ax.set_ylabel(unit)
                ax.grid(alpha=0.15)
        axes[0].set_ylim(tc_min, tc_max)
        axes[1].set_ylim(0, 105)
        axes[0].legend(loc="upper left", fontsize=8)
        axes[1].set_xlabel("첫 온도 관측 후 경과시간 (분) · 선: 1분 평균 · 옅은 색: 그 분의 최소~최대")
        fig.suptitle(f"{grade} | Z{zone} 온도·출력 | {message}")
        output = FIGURES / f"Z{zone}_온도_출력" / "강종별"
        output.mkdir(parents=True, exist_ok=True)
        fig.savefig(output / f"{grade}_Z{zone}_온도_출력.png", dpi=140)
        if grade == VIEW_GRADE:
            plt.show()
        plt.close(fig)

    # CP도 같은 색과 경과시간으로 비교하기 위함
    fig, ax = plt.subplots(figsize=(13, 4), layout="constrained")
    for index, (lot, label) in enumerate(zip(lot_ids, legend_labels)):
        color = COLORS[index % len(COLORS)]
        mean, low, high = means[lot], lows[lot], highs[lot]
        elapsed = (mean.index - lots.loc[lot, "시작"]).total_seconds() / 60
        ax.plot(elapsed, mean["CP-Z4"], color=color, linewidth=1, label=label)
        ax.fill_between(elapsed, low["CP-Z4"], high["CP-Z4"], color=color, alpha=0.08)
    ax.set_title(f"{grade} | CP-Z4 | {message}")
    ax.set_ylabel("CP-Z4 기록값 (단위 미확인)")
    ax.set_xlabel("첫 온도 관측 후 경과시간 (분) · 선: 1분 평균 · 옅은 색: 그 분의 최소~최대")
    ax.grid(alpha=0.15)
    ax.legend(loc="upper left", fontsize=8)
    output = FIGURES / "CP_기록값" / "강종별"
    output.mkdir(parents=True, exist_ok=True)
    fig.savefig(output / f"{grade}_CP_Z4.png", dpi=140)
    if grade == VIEW_GRADE:
        plt.show()
    plt.close(fig)


## 3. 그림을 읽고 기록할 내용

- 온도는 비슷한데 출력 수준이 다른 LOT가 있는지 보기.
- Z3·Z4·Z5가 함께 변하는지, 특정 구역만 변하는지 보기.
- CP 변화가 온도·출력 변화와 같은 때인지 보기. 같은 때라는 사실만으로 원인을 정하지 않음.
- 한 LOT에서만 보이는 모습과 같은 강종의 여러 LOT에서 반복되는 모습을 구분하기.

**이 그림은 팀에서 정한 7개 예측 대상의 특징을 설명하기 위한 것임. 7개가 다른 모든 채널보다 좋은 예측 대상이라는 사실을 증명하는 것은 아님.** 높은 출력이나 선의 차이를 곧바로 정상·이상 정답으로 만들지 않음.

038·114의 SPCEN 비교에서는 038의 높은 OP-Z4를 유지함. 058의 정지와 061의 재가동은 팀 확정 해석을 유지함. 단일 LOT 강종은 강종 특성과 해당 LOT의 운전 상태를 분리하기 어려움.

LOT별 관찰과 같은 강종 비교를 확인한 뒤 [05번 구간 검토](05_interval_review.ipynb)에서 제외 구간과 라벨 근거를 함께 확인함.


## 4. SPHC 마지막 LOT의 TC-Z7·TC-Z8 수준 다시 보기

앞의 비교 대상 7개에는 TC-Z7·TC-Z8이 없었음. 같은 SPHC의 마지막 LOT에서 TC-Z7이 높다는 관찰을 빠뜨리지 않도록 과시효 구역을 추가로 확인함.

SPHC는 실제 관측 시간순으로 **240044 → 240061 → 240069 → 240091**임. [05번](05_interval_review.ipynb)과 같은 사용자 지정 제외 구간을 먼저 적용하며, 여기서는 061의 02:30 이전 승온 구간이 빠짐. 다른 강종을 정상 기준으로 섞지 않음.

표의 평균·중앙값·5~95백분위·최솟값·최댓값은 남은 원본 관측으로 계산함. 아래 선은 보기 편하게 LOT별 1분 평균을 그린 것임. 이전 세 LOT도 정상으로 확정된 표본이라는 뜻은 아님.


In [ ]:
from namelesspark_pipeline import apply_user_exclusions, build_maintenance_context

review_raw, _ = apply_user_exclusions(raw)
sphc_lots = lots.loc[lots["강종"].eq("SPHC")].sort_values("시작")
sphc = review_raw.loc[
    ~review_raw["EXCLUDED"] & review_raw["LOT_NO"].isin(sphc_lots.index)
].copy()
aging_channels = ["TC-Z7", "TC-Z8", "TC-Z9", "OP-Z7", "OP-Z8"]
review_output = ROOT / "data" / "processed" / "analysis" / "grade" / "v1"
review_output.mkdir(parents=True, exist_ok=True)
aging_figures = FIGURES / "과시효_추가검토"
aging_figures.mkdir(parents=True, exist_ok=True)

summary_rows = []
for lot, rows in sphc.groupby("LOT_NO", sort=True):
    for channel in aging_channels:
        values = rows[channel].dropna()
        summary_rows.append({
            "LOT_NO": lot, "CHANNEL": channel, "COUNT": len(values),
            "MEAN": values.mean(), "MEDIAN": values.median(),
            "Q05": values.quantile(0.05), "Q95": values.quantile(0.95),
            "MIN": values.min(), "MAX": values.max(),
        })
sphc_aging_summary = pd.DataFrame(summary_rows)
sphc_aging_summary.to_csv(
    review_output / "04_sphc_aging_summary.csv", index=False, encoding="utf-8-sig"
)
display(sphc_aging_summary.loc[
    sphc_aging_summary["CHANNEL"].eq("TC-Z7")
].round(3))
display(sphc_aging_summary.pivot(
    index="LOT_NO", columns="CHANNEL", values="MEDIAN"
).round(3))

# 같은 강종의 각 LOT를 따로 그림. 잘린 061도 원래 LOT 시작 기준 경과시간을 유지함.
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, layout="constrained")
for lot, color in zip(sphc_lots.index, COLORS):
    minute = sphc.loc[sphc["LOT_NO"].eq(lot)].set_index("MEAS_DT")[
        aging_channels
    ].resample("1min").mean()
    elapsed = (minute.index - sphc_lots.loc[lot, "시작"]).total_seconds() / 60
    label = f"{lot} ({sphc_lots.loc[lot, '시작']:%m/%d})"
    for ax, channel in zip(axes.flat, ["TC-Z7", "TC-Z8", "OP-Z7", "OP-Z8"]):
        ax.plot(elapsed, minute[channel], color=color, linewidth=1, label=label)
        ax.set_title(channel)
        ax.set_ylabel("온도 (℃)" if channel.startswith("TC") else "출력 (%)")
        ax.grid(alpha=0.15)
        ax.legend(fontsize=8)
for ax in axes[0]:
    ax.set_ylim(sphc[["TC-Z7", "TC-Z8"]].min().min() - 0.2,
                sphc[["TC-Z7", "TC-Z8"]].max().max() + 0.2)
for ax in axes[1]:
    ax.set_ylim(sphc[["OP-Z7", "OP-Z8"]].min().min() - 1,
                sphc[["OP-Z7", "OP-Z8"]].max().max() + 1)
    ax.set_xlabel("원래 LOT 첫 관측 후 경과시간 (분)")
fig.suptitle("SPHC 시간순 비교 · 지정 구간 제외 후 · 선: 1분 평균")
fig.savefig(aging_figures / "SPHC_TC_Z7_Z8_비교.png", dpi=140)
plt.show()
plt.close(fig)


### 마지막 240091의 관측 안에서 갑자기 오른 것인지 보기

TC-Z7 중앙값은 앞의 세 LOT에서 약 **329.74~329.90℃**, 마지막 240091에서 **332.963℃**임. TC-Z8도 240091에서 **333.295℃**로, 앞의 약 **329.73~330.29℃**보다 높음. 반면 OP-Z7·OP-Z8 중앙값은 직전 240069와 비슷한 수준이고 TC-Z9의 중앙값도 약 329℃대임.

240091은 **09/08 22:03 첫 관측부터 높은 수준**이며, 관측 안에서 330℃에서 333℃로 갑자기 오르는 형태는 아님. TC-Z7은 첫 30분 중앙값 332.920℃, 마지막 30분 332.923℃임. 이전 LOT와 관측 날짜가 떨어져 있으므로 높은 수준으로 바뀐 실제 시점은 이 자료만으로 알 수 없음.


In [ ]:
last_sphc_lot = int(sphc_lots.index[-1])
last_sphc = sphc.loc[sphc["LOT_NO"].eq(last_sphc_lot)].sort_values("MEAS_DT")
last_start = last_sphc["MEAS_DT"].min()
last_end = last_sphc["MEAS_DT"].max()

segment_rows = []
for name, mask in [
    ("첫 30분", last_sphc["MEAS_DT"].lt(last_start + pd.Timedelta(minutes=30))),
    ("마지막 30분", last_sphc["MEAS_DT"].ge(last_end - pd.Timedelta(minutes=30))),
]:
    for channel in aging_channels:
        values = last_sphc.loc[mask, channel].dropna()
        segment_rows.append({
            "LOT_NO": last_sphc_lot, "SEGMENT": name, "CHANNEL": channel,
            "COUNT": len(values), "MEDIAN": values.median(),
            "MIN": values.min(), "MAX": values.max(),
        })
segment_summary = pd.DataFrame(segment_rows)
display(segment_summary.pivot(
    index="SEGMENT", columns="CHANNEL", values="MEDIAN"
).round(3))
segment_summary.to_csv(
    review_output / "04_sphc_last_lot_segments.csv", index=False, encoding="utf-8-sig"
)

last_minutes = last_sphc.set_index("MEAS_DT")[aging_channels].resample("1min").mean()
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, layout="constrained")
for ax, channels, unit in zip(
    axes, [["TC-Z7", "TC-Z8"], ["OP-Z7", "OP-Z8"], ["TC-Z9"]],
    ["온도 (℃)", "출력 (%)", "온도 (℃)"],
):
    for channel in channels:
        ax.plot(last_minutes.index, last_minutes[channel], linewidth=1, label=channel)
    ax.set_ylabel(unit)
    ax.grid(alpha=0.15)
    ax.legend(loc="upper left")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
axes[-1].set_xlabel("2024년 실제 측정 날짜·시각 · 선: 1분 평균")
fig.suptitle(f"SPHC 마지막 LOT {last_sphc_lot} · 시작부터 높게 관측된 Z7·Z8")
fig.savefig(aging_figures / f"SPHC_{last_sphc_lot}_과시효_시계열.png", dpi=140)
plt.show()
plt.close(fig)


### 정비·부품교체 기록과 함께 보기

관측 기간과 겹친 CAL01 정비는 없음. 바로 전 완료 정비와 관측 종료 뒤 BM·TBM·CBM별 첫 감지 기록을 아래에 표시함. 시간상 앞뒤에 있는 기록을 확인하는 표이며 원인을 연결한 것은 아님.

- 직전 완료 정비는 08/27~08/28 BM의 **모터 과열** 기록이며, 오일씰 1개 교체가 연결됨.
- 09/11 12:55 BM의 **절연 저하** 감지는 240091 관측 종료 약 **52.18시간 뒤**임. 부품교체 기록은 없음.
- 뒤의 TBM은 **정기점검 - 이상없음**, CBM은 **재점검 필요**이며 둘 다 부품교체 기록은 없음.
- 정기수리 캘린더도 240091 관측과 겹치지 않음. 이 기록만으로 Z7·Z8의 높은 수준이 절연 저하 때문이라고 할 수 없음.

**같은 강종의 이전 관측과 다른 지속적인 수준 차이는 검토할 근거임. 다만 설정온도·운전 조건·센서 교정 기록이 없어 현재 단계에서 고장이나 이상 라벨 1로 확정하지 않음.** 이 관찰은 [05번 구간 검토](05_interval_review.ipynb)에서 다른 근거와 함께 판단할 항목임.


In [ ]:
maintenance_context, _ = build_maintenance_context(
    sources["maint"], sources["partchg"], sources["part"]
)
previous_work = maintenance_context.loc[
    maintenance_context["END_DT"].lt(last_start)
].sort_values("END_DT").tail(1).assign(CONTEXT="직전 완료 정비")
following_work = maintenance_context.loc[
    maintenance_context["DETC_DT"].gt(last_end)
].sort_values("DETC_DT").groupby("MNT_TYPE", sort=False).head(1).assign(
    CONTEXT="관측 종료 뒤 유형별 첫 감지"
)
nearby_work = pd.concat([previous_work, following_work]).sort_values("DETC_DT")
nearby_work["DETECTION_HOURS_FROM_LOT_END"] = (
    nearby_work["DETC_DT"] - last_end
).dt.total_seconds() / 3600
maintenance_columns = [
    "CONTEXT", "WO_NO", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT",
    "REMARK", "PART_CHG", "PART_NAMES", "PART_QTY_TOTAL",
    "DETECTION_HOURS_FROM_LOT_END",
]
display(nearby_work[maintenance_columns])
nearby_work[maintenance_columns].to_csv(
    review_output / "04_sphc_last_lot_maintenance.csv",
    index=False, encoding="utf-8-sig",
)

overlapping_work = maintenance_context.loc[
    maintenance_context["DETC_DT"].between(last_start, last_end)
    | (maintenance_context["STRT_DT"].le(last_end)
       & maintenance_context["END_DT"].ge(last_start))
]
calendar = sources["repair"].loc[sources["repair"]["PLANT_CD"].eq("CR1")]
overlapping_plan = calendar.loc[
    calendar["STRT_DT"].le(last_end)
    & (calendar["END_DT"] + pd.Timedelta(days=1)).gt(last_start)
]
print("관측과 감지·작업이 겹친 CAL01 정비 건수:", len(overlapping_work))
print("관측과 겹친 CR1 정기수리 일정 수:", len(overlapping_plan))
